## Exploratory Data Analysis and Cleanup for Children's Books

In [1]:
# importing libraries
import pandas as pd
pd.set_option("display.max_columns", None)
import os
from dotenv import load_dotenv
import numpy as np
from langdetect import detect, DetectorFactory
from tqdm.auto import tqdm
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ast
from collections import OrderedDict
import re

/Users/navyajain/Downloads/build/agentic-book-recommendation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DetectorFactory.seed = 42

In [3]:
# paths
load_dotenv()
#children_path = os.getenv("CHILDREN_BOOKS")

True

In [4]:
#print(children_path)

In [5]:
comic_df = pd.read_json('../data/books/goodreads_books_romance.json', lines=True)

In [6]:
comic_df.shape

(335449, 29)

In [7]:
comic_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,,4,[],US,,"[{'count': '4', 'name': 'to-read'}, {'count': ...",,true,3.86,,[],Secrets. Sometimes keeping them in confidence ...,ebook,https://www.goodreads.com/book/show/34883016-p...,"[{'author_id': '5807700', 'role': ''}]",Gone Writing Publishing,,3,9781370889471,5,,2017,https://www.goodreads.com/book/show/34883016-p...,https://images.gr-assets.com/books/1493525974m...,34883016,5,56135087,Playmaker: A Venom Series Novella,Playmaker: A Venom Series Novella
1,,21,[811663],US,en-US,"[{'count': '598', 'name': 'to-read'}, {'count'...",B01BLJGA9S,true,4.23,B01BLJGA9S,"[25515353, 20483269, 25650829, 18913492, 22578...",,,https://www.goodreads.com/book/show/29074693-p...,"[{'author_id': '5360266', 'role': ''}]",,,,,,,,https://www.goodreads.com/book/show/29074693-p...,https://s.gr-assets.com/assets/nophoto/book/11...,29074693,149,46079519,"Prowled Darkness (Dante's Circle, #7)","Prowled Darkness (Dante's Circle, #7)"
2,1597371289,8,[],US,eng,"[{'count': '16215', 'name': 'classics'}, {'cou...",,false,3.99,B0083Z3O8Y,"[31242, 374380, 20564, 383206, 7891, 6335178, ...",The funny and heartwarming story of a young la...,Audio CD,https://www.goodreads.com/book/show/3209316-emma,"[{'author_id': '1265', 'role': ''}]",Brilliance Audio,544,25,9781597371285,9,,2005,https://www.goodreads.com/book/show/3209316-emma,https://s.gr-assets.com/assets/nophoto/book/11...,3209316,42,3360164,Emma,Emma
3,,27,[938303],US,en-GB,"[{'count': '25', 'name': 'to-read'}, {'count':...",B01HX6PENG,true,4.31,B01HX6PENG,[],"In the Finding Fatherhood series, these shifte...",,https://www.goodreads.com/book/show/30838933-g...,"[{'author_id': '90411', 'role': ''}, {'author_...",,,,,,,,https://www.goodreads.com/book/show/30838933-g...,https://s.gr-assets.com/assets/nophoto/book/11...,30838933,139,51437308,"Guardian Cougar (Finding Fatherhood, #2)","Guardian Cougar (Finding Fatherhood, #2)"
4,,15,[],US,eng,"[{'count': '1492', 'name': 'to-read'}, {'count...",B013Q70BQG,true,3.98,B013Q70BQG,"[386097, 7774253, 576667, 471907, 43174, 38977...",You've Got Mailmeets Julie & Juliain the new f...,,https://www.goodreads.com/book/show/27419760-w...,"[{'author_id': '47231', 'role': ''}]",,,,,,,,https://www.goodreads.com/book/show/27419760-w...,https://images.gr-assets.com/books/1457306424m...,27419760,167,46003673,Wedding Girl,Wedding Girl


In [8]:
df = comic_df.copy()

| Column | Meaning |
|---|---|
| `isbn` | 10-digit ISBN for the book edition, if available. |
| `text_reviews_count` | Number of written/text reviews for this book edition. |
| `series` | List of Goodreads series IDs the book belongs to. Empty list means no series or unavailable. |
| `country_code` | Country code from Goodreads metadata, often `US`. |
| `language_code` | Language code for the edition, such as `eng`, `en-US`, etc. Can be blank. |
| `popular_shelves` | User-created Goodreads shelf tags for the book, with shelf name and count. |
| `asin` | Amazon Standard Identification Number, if available. |
| `is_ebook` | Whether the edition is an ebook, stored as `"true"` or `"false"`. |
| `average_rating` | Average Goodreads rating for the book, usually from 1 to 5. |
| `kindle_asin` | Kindle-specific Amazon ID, if available. |
| `similar_books` | List of Goodreads `book_id`s marked as similar books. |
| `description` | Book description or blurb text. Can be empty. |
| `format` | Book format, such as `Hardcover`, `Paperback`, or `Kindle Edition`. |
| `link` | Goodreads link for the book page. |
| `authors` | List of author objects, usually containing `author_id` and `role`. |
| `publisher` | Publisher name for this edition. |
| `num_pages` | Number of pages, if available. |
| `publication_day` | Publication day of the month, if available. |
| `isbn13` | 13-digit ISBN for the book edition, if available. |
| `publication_month` | Publication month, if available. |
| `edition_information` | Extra edition information, such as `Anniversary Edition` or `First Edition`. |
| `publication_year` | Publication year for this edition. |
| `url` | Goodreads URL for the book page. Usually similar to `link`. |
| `image_url` | URL for the book cover image. |
| `book_id` | Goodreads ID for this specific book edition. Useful for joining with reviews/interactions. |
| `ratings_count` | Number of ratings for this book edition. |
| `work_id` | Goodreads work ID. Groups multiple editions of the same underlying book. Useful for deduplication. |
| `title` | Full book title as shown on Goodreads, often including series information. |
| `title_without_series` | Book title with series information removed. Better for clean display/search. |

In [9]:
df.shape

(335449, 29)

In [10]:
df.columns

Index(['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin',
       'similar_books', 'description', 'format', 'link', 'authors',
       'publisher', 'num_pages', 'publication_day', 'isbn13',
       'publication_month', 'edition_information', 'publication_year', 'url',
       'image_url', 'book_id', 'ratings_count', 'work_id', 'title',
       'title_without_series'],
      dtype='str')

In [11]:
df = df[['title', 'title_without_series', 'series', 'country_code', 'language_code',
       'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id', 'book_id',
       'text_reviews_count', 'average_rating', 'ratings_count', 'popular_shelves',
       'is_ebook', 'format', 'num_pages', 'publisher', 'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [12]:
df.shape

(335449, 26)

In [13]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,Playmaker: A Venom Series Novella,Playmaker: A Venom Series Novella,[],US,,,,,9781370889471,56135087,34883016,4,3.86,5,"[{'count': '4', 'name': 'to-read'}, {'count': ...",true,ebook,,Gone Writing Publishing,3,5,2017,,Secrets. Sometimes keeping them in confidence ...,"[{'author_id': '5807700', 'role': ''}]",[]
1,"Prowled Darkness (Dante's Circle, #7)","Prowled Darkness (Dante's Circle, #7)",[811663],US,en-US,,B01BLJGA9S,B01BLJGA9S,,46079519,29074693,21,4.23,149,"[{'count': '598', 'name': 'to-read'}, {'count'...",true,,,,,,,,,"[{'author_id': '5360266', 'role': ''}]","[25515353, 20483269, 25650829, 18913492, 22578..."
2,Emma,Emma,[],US,eng,1597371289,,B0083Z3O8Y,9781597371285,3360164,3209316,8,3.99,42,"[{'count': '16215', 'name': 'classics'}, {'cou...",false,Audio CD,544,Brilliance Audio,25,9,2005,,The funny and heartwarming story of a young la...,"[{'author_id': '1265', 'role': ''}]","[31242, 374380, 20564, 383206, 7891, 6335178, ..."
3,"Guardian Cougar (Finding Fatherhood, #2)","Guardian Cougar (Finding Fatherhood, #2)",[938303],US,en-GB,,B01HX6PENG,B01HX6PENG,,51437308,30838933,27,4.31,139,"[{'count': '25', 'name': 'to-read'}, {'count':...",true,,,,,,,,"In the Finding Fatherhood series, these shifte...","[{'author_id': '90411', 'role': ''}, {'author_...",[]
4,Wedding Girl,Wedding Girl,[],US,eng,,B013Q70BQG,B013Q70BQG,,46003673,27419760,15,3.98,167,"[{'count': '1492', 'name': 'to-read'}, {'count...",true,,,,,,,,You've Got Mailmeets Julie & Juliain the new f...,"[{'author_id': '47231', 'role': ''}]","[386097, 7774253, 576667, 471907, 43174, 38977..."


In [14]:
df['clean_title'] = df['title_without_series']

In [15]:
df.shape

(335449, 27)

### Detecting languages

In [16]:
def detect_lang(text):
    if pd.isna(text) or len(str(text).strip()) < 30:
        return np.nan

    try:
        lang = detect(str(text))
        return "eng" if lang == "en" else lang
    except:
        return np.nan

In [17]:
# normalize empty strings to NaN
df["language_code_final"] = df["language_code"].replace("", np.nan)

# create title + description text
df["language_text"] = (
    df["clean_title"].fillna("") + " " + df["description"].fillna("")
)

# detect only missing language codes
missing_lang_mask = df["language_code_final"].isna()

texts_to_detect = df.loc[missing_lang_mask, "language_text"]

detected_languages = [
    detect_lang(text)
    for text in tqdm(texts_to_detect, desc="Detecting languages")
]

df.loc[missing_lang_mask, "detected_language_code"] = detected_languages

# fill missing language code with detected language
df["language_code_final"] = df["language_code_final"].fillna(
    df["detected_language_code"]
)

# normalize English variants
df["language_code_final"] = (
    df["language_code_final"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "en": "eng",
        "en-us": "eng",
        "en-gb": "eng",
        "en-ca": "eng"
    })
)

Detecting languages: 100%|██████████| 104396/104396 [04:36<00:00, 377.08it/s]


In [18]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,clean_title,language_code_final,language_text,detected_language_code
0,Playmaker: A Venom Series Novella,Playmaker: A Venom Series Novella,[],US,,,,,9781370889471,56135087,34883016,4,3.86,5,"[{'count': '4', 'name': 'to-read'}, {'count': ...",true,ebook,,Gone Writing Publishing,3,5,2017,,Secrets. Sometimes keeping them in confidence ...,"[{'author_id': '5807700', 'role': ''}]",[],Playmaker: A Venom Series Novella,eng,Playmaker: A Venom Series Novella Secrets. Som...,eng
1,"Prowled Darkness (Dante's Circle, #7)","Prowled Darkness (Dante's Circle, #7)",[811663],US,en-US,,B01BLJGA9S,B01BLJGA9S,,46079519,29074693,21,4.23,149,"[{'count': '598', 'name': 'to-read'}, {'count'...",true,,,,,,,,,"[{'author_id': '5360266', 'role': ''}]","[25515353, 20483269, 25650829, 18913492, 22578...","Prowled Darkness (Dante's Circle, #7)",eng,"Prowled Darkness (Dante's Circle, #7)",nan
2,Emma,Emma,[],US,eng,1597371289,,B0083Z3O8Y,9781597371285,3360164,3209316,8,3.99,42,"[{'count': '16215', 'name': 'classics'}, {'cou...",false,Audio CD,544,Brilliance Audio,25,9,2005,,The funny and heartwarming story of a young la...,"[{'author_id': '1265', 'role': ''}]","[31242, 374380, 20564, 383206, 7891, 6335178, ...",Emma,eng,Emma The funny and heartwarming story of a you...,nan
3,"Guardian Cougar (Finding Fatherhood, #2)","Guardian Cougar (Finding Fatherhood, #2)",[938303],US,en-GB,,B01HX6PENG,B01HX6PENG,,51437308,30838933,27,4.31,139,"[{'count': '25', 'name': 'to-read'}, {'count':...",true,,,,,,,,"In the Finding Fatherhood series, these shifte...","[{'author_id': '90411', 'role': ''}, {'author_...",[],"Guardian Cougar (Finding Fatherhood, #2)",eng,"Guardian Cougar (Finding Fatherhood, #2) In th...",nan
4,Wedding Girl,Wedding Girl,[],US,eng,,B013Q70BQG,B013Q70BQG,,46003673,27419760,15,3.98,167,"[{'count': '1492', 'name': 'to-read'}, {'count...",true,,,,,,,,You've Got Mailmeets Julie & Juliain the new f...,"[{'author_id': '47231', 'role': ''}]","[386097, 7774253, 576667, 471907, 43174, 38977...",Wedding Girl,eng,Wedding Girl You've Got Mailmeets Julie & Juli...,nan


In [19]:
df.columns

Index(['title', 'title_without_series', 'series', 'country_code',
       'language_code', 'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id',
       'book_id', 'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'is_ebook', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'clean_title', 'language_code_final', 'language_text',
       'detected_language_code'],
      dtype='str')

In [20]:
df.shape

(335449, 30)

In [21]:
df=df[['clean_title', 'series', 'language_code_final', 
       'isbn', 'isbn13', 'work_id', 'book_id', 
       'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [22]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,Playmaker: A Venom Series Novella,[],eng,,9781370889471,56135087,34883016,4,3.86,5,"[{'count': '4', 'name': 'to-read'}, {'count': ...",ebook,,Gone Writing Publishing,3,5,2017,,Secrets. Sometimes keeping them in confidence ...,"[{'author_id': '5807700', 'role': ''}]",[]
1,"Prowled Darkness (Dante's Circle, #7)",[811663],eng,,,46079519,29074693,21,4.23,149,"[{'count': '598', 'name': 'to-read'}, {'count'...",,,,,,,,,"[{'author_id': '5360266', 'role': ''}]","[25515353, 20483269, 25650829, 18913492, 22578..."
2,Emma,[],eng,1597371289,9781597371285,3360164,3209316,8,3.99,42,"[{'count': '16215', 'name': 'classics'}, {'cou...",Audio CD,544,Brilliance Audio,25,9,2005,,The funny and heartwarming story of a young la...,"[{'author_id': '1265', 'role': ''}]","[31242, 374380, 20564, 383206, 7891, 6335178, ..."
3,"Guardian Cougar (Finding Fatherhood, #2)",[938303],eng,,,51437308,30838933,27,4.31,139,"[{'count': '25', 'name': 'to-read'}, {'count':...",,,,,,,,"In the Finding Fatherhood series, these shifte...","[{'author_id': '90411', 'role': ''}, {'author_...",[]
4,Wedding Girl,[],eng,,,46003673,27419760,15,3.98,167,"[{'count': '1492', 'name': 'to-read'}, {'count...",,,,,,,,You've Got Mailmeets Julie & Juliain the new f...,"[{'author_id': '47231', 'role': ''}]","[386097, 7774253, 576667, 471907, 43174, 38977..."


In [23]:
df.shape

(335449, 21)

In [24]:
df = df[
    df["language_code_final"].astype("string").str.strip().str.lower() == "eng"
].copy()

In [25]:
df.shape

(288276, 21)

In [26]:
df['format'].value_counts().shape

(225,)

In [27]:
unique_formats = df["format"].dropna().unique().tolist()
print(unique_formats)

['ebook', '', 'Audio CD', 'Kindle Edition', 'Paperback', 'Paperback Manga', 'Audible Audio', 'Trade Paperback', 'Mass Market Paperback', 'Audio', 'Hardcover', 'Audiobook', 'Paperback and eBook', 'online fanfiction', 'ebook and paperback option', 'online fiction', 'Unknown Binding', 'Nook', 'Large print', 'MP3 CD', 'Paperback (Large Print)', 'Online Fiction - Complete', 'Short Story', "Audio CD's", 'Wattpad', 'NOOK ebook', 'Library Binding', 'e-book, digital format', 'paperback', 'free online fiction', 'free manga scanlation', 'Paperback, ebook', 'audio', 'Audio Cassette', 'Leather Bound', 'E-book', 'Free Online Fiction', 'Online', 'e book', 'Free Online Read', 'paperback and ebook', 'Free online read', 'Online Fiction', 'Tradepaperback', 'Paperback and ebook', 'Kindle Edition, Paperback', 'iBooks', 'Harlequin Online Read', 'Kindle/Paperback', 'Advanced Review Copy', 'Newsletter Serial/Wattpad Story', 'comics', 'Unbound', 'Box Set', 'Paperback &amp; eBook', 'Mass paperback', 'Mass Marke

In [28]:
# 1. Define readable formats to keep
keep_formats = {
    # Standard print
    "Paperback",
    "paperback",
    "Hardcover",
    "hardcover",
    "Hardback",
    "Trade Paperback",
    "trade paperback",
    "Trade paperback",
    "Trade Paper",
    "Tradepaperback",
    "Mass Market Paperback",
    "mass market paperback",
    "Mass paperback",
    "Mass Market",
    "Mass Paper Back",
    "Pocket Paperback",
    "Pocket",
    "Perfect Paperback",
    "Oversized Paperback",
    "Paperback (Large Print)",
    "Paperback - Large Print",
    "Large print",
    "large print",
    "Large Type",
    "Hardcover - Large Print",
    "Hardcover (Large Print)",
    "Hardcover [LARGE PRINT]",
    "Library Binding",
    "Leather Bound",
    "Spiral-bound",
    "Flexibound",
    "Unbound",
    "cloth",
    "Softcover",
    "softcover",
    "Soft Cover",
    "Board book",
    "Book",
    "book",
    "Print",
    "print",
    "Print on Demand",
    "Book Club Omnibus",
    "Box Set",
    "ARC",
    "Advanced Review Copy",

    # Manga / comics / graphic / webcomic
    "Paperback Manga",
    "Manga",
    "manga",
    "Comic",
    "comics",
    "Webcomic",
    "Webcomic/Manhua",
    "Web Comic/ Manhua",
    "Webtoon",
    "webtoon",

    # Ebooks / digital readable text
    "ebook",
    "Ebook",
    "eBook",
    "E-book",
    "e book",
    "(e-book)",
    "epub",
    "Mobi",
    "PDF",
    "pdf",
    "eBook (PDF)",
    "PDF (Movie Script)",
    "Digital Copy",
    "digital",
    "downloadable e-book",
    "MultiFormat eBook",
    "e-book, digital format",
    "APP Exclusive ebook",
    "Website ebook",
    "Kindle Edition",
    "Kindle",
    "Kindle edition",
    "Nook",
    "Nook Book",
    "Nook Edition",
    "NOOK ebook",
    "Kobo",
    "Kobo ebook",
    "iBooks",
    "ibook",
    "Scribd",

    # Mixed print + ebook
    "Paperback and eBook",
    "Paperback and Ebook",
    "Paperback and ebook",
    "paperback and ebook",
    "Paperback, ebook",
    "paperback, Ebook",
    "Paperback, Digital",
    "Paperback &amp; eBook",
    "Paperback &amp; ebook",
    "ebook &amp; paperback",
    "Ebook &amp; Paperback",
    "ebook and paperback",
    "ebook and Paperback",
    "Ebook and paperback",
    "ebook and paperback option",
    "ebook, paperback",
    "ebook and print",
    "ebook and Print",
    "E-book and Print",
    "e-book &amp; print",
    "ebook &amp; print",
    "e book Print",
    "Print and ebook",
    "ebook, print",
    "Paperback/Kindle",
    "Kindle/Paperback",
    "Kindle, Nook, Paperback",
    "Kindle Edition, Paperback",
    "Kindle Edition / Paperback",
    "Kindle Edition/ ebook",
    "Paperback, ebook, KU",
    "ebook, kindle",
    "Pdf e-book and Paperback",
    "paperback, pdf, mobipocket, html ebook",
    "pdf, html,microsoft reader, mobipocket",
    "Electronic, Paperback",

    # Online readable fiction / web serials / fanfiction
    "Online",
    "online",
    "on-line",
    "Online Read",
    "online read",
    "Online read",
    "Free Online Read",
    "Free online read",
    "Free Read Online",
    "Harlequin Online Read",
    "Online Fiction",
    "online fiction",
    "Online fiction",
    "On-Line fiction",
    "Online Fiction - Complete",
    "Free Online Fiction",
    "free online fiction",
    "Free online fiction",
    "Online Free Fiction",
    "Original Online Fiction",
    "original online fiction",
    "Online Original Fictions",
    "free online original fiction",
    "free online story",
    "Free online read",
    "free online",
    "free ebook",
    "Online Publication",
    "Online Publication, Free",
    "Online Freebie",
    "Online Novella",
    "online serial",
    "Online Serial",
    "Serial Fiction",
    "web novel",
    "Wattpad",
    "Patreon Edition",
    "Newsletter Serial/Wattpad Story",
    "Newsletter Serial",
    "Online Journal",
    "Online Journal Archive",
    "online article",
    "Fanfiction",
    "fanfiction",
    "Online Fanfiction",
    "online fanfiction",
    "https://www.wattpad.com/story/18646750",
    "https://www.wattpad.com/story/49206761-mated-to-the-werewolf-king-now-completed",

    # Short fiction / genre labels that are still readable
    "Short Story",
    "Novella",
    "Ebook Novella",
    "e-book, novella",
    "Romance",
    "Magazine",

    # Foreign print labels
    "Poche",
    "Broche",
    "Broschiert",
}

drop_formats = {
    # Missing / unknown
    "",
    "Unknown Binding",
    "Other Format",

    # Audio
    "Audio",
    "audio",
    "Audio CD",
    "Audio CD (Abr.)",
    "Audio CD (Unabridged)",
    "Audio CD (Unabr.)",
    "Audio CD - Abr",
    "Audio Cassette",
    "Audio Cassette (Libr.) (Unabr.)",
    "Audible Audio",
    "Audible Download",
    "Audible, USA Download",
    "Audiobook",
    "Audiobook, Unabridged",
    "Audiobook Unabridged",
    "Audiobook, Unabridged Audiobook (19Hrs 45Min)",
    "Unabridged Audiobook",
    "eAudiobook",
    "Audio Download",
    "eAudio",
    "MP3 CD",
    "MP3",
    "MP3 Book",
    "Playaway",
    "Podcast",
    "Book on CD",
    "Unabridged CD",

    # Mixed with audio; drop for now
    "Paperback + Audio CD",

    # Pirated / scanlation source
    "free manga scanlation",
    "English scanlation",

    # Video / software / non-book media
    "CD-ROM",
    "Video Book",

    # Too vague / not actual format
    "Marriage Counseling",
    "Penguin Classics",
}


# 4. Clean the format column
df["format_clean"] = (df["format"].fillna("").astype(str).str.strip())

# 5. Assign each row to a format group
df["format_group"] = np.select(
    [
        df["format_clean"].isin(keep_formats),
        df["format_clean"].isin(drop_formats)
    ],
    ["text", "other"], default="review")

# 6. Check any unclassified formats
df["format_group"].value_counts()

format_group
text      180943
other     107309
review        24
Name: count, dtype: int64

In [29]:
df = df[df["format_group"] == "text"].copy()

In [30]:
df["format_group"].value_counts()

format_group
text    180943
Name: count, dtype: int64

In [31]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,format_clean,format_group
0,Playmaker: A Venom Series Novella,[],eng,,9781370889471,56135087,34883016,4,3.86,5,"[{'count': '4', 'name': 'to-read'}, {'count': ...",ebook,,Gone Writing Publishing,3,5,2017,,Secrets. Sometimes keeping them in confidence ...,"[{'author_id': '5807700', 'role': ''}]",[],ebook,text
9,"Healer's Touch (Hearts And Thrones, #4)",[709800],eng,,,43219002,23615331,11,3.75,112,"[{'count': '153', 'name': 'to-read'}, {'count'...",Kindle Edition,279,Amy Raby,23,11,2014,1st Edition,Marius believes himself to be an ordinary smal...,"[{'author_id': '6462472', 'role': ''}]","[15738275, 23557009, 17225973, 15750539, 23996...",Kindle Edition,text
10,"Nemesis (Southern Comfort, #4)",[521302],eng,,,25481836,18138604,10,4.09,162,"[{'count': '647', 'name': 'to-read'}, {'count'...",Kindle Edition,524,,1,10,2013,,Chosen as one of the Best Reads of 2013 by Kin...,"[{'author_id': '6480066', 'role': ''}]","[21917872, 25698219, 17068443, 16005563, 16299...",Kindle Edition,text
14,Love Means... Patience,[629995],eng,1627988912,9781627988919,41377166,22499086,1,4.11,6,"[{'count': '201', 'name': 'to-read'}, {'count'...",Paperback,200,Dreamspinner Press,23,5,2014,,A Love Means... Story\nYears after his dischar...,"[{'author_id': '2953781', 'role': ''}]","[18386330, 22072041, 22041415, 21454390, 25278...",Paperback,text
15,"A.I. Revolution, Vol. 1",[910384],eng,1933617640,9781933617640,2256459,2250580,7,3.44,46,"[{'count': '15', 'name': 'to-read'}, {'count':...",Paperback Manga,206,Go! Comi,2,1,2007,,"Like everyone else in the future, Sui's used t...","[{'author_id': '1015982', 'role': ''}]",[],Paperback Manga,text


In [32]:
df.shape

(180943, 23)

In [33]:
df.columns

Index(['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'format_clean', 'format_group'],
      dtype='str')

In [34]:
df = df[['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format_clean', 'num_pages', 'publisher',
       'publication_year', 'description', 'authors', 'similar_books']]

In [35]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format_clean,num_pages,publisher,publication_year,description,authors,similar_books
0,Playmaker: A Venom Series Novella,[],eng,,9781370889471,56135087,34883016,4,3.86,5,"[{'count': '4', 'name': 'to-read'}, {'count': ...",ebook,,Gone Writing Publishing,2017,Secrets. Sometimes keeping them in confidence ...,"[{'author_id': '5807700', 'role': ''}]",[]
9,"Healer's Touch (Hearts And Thrones, #4)",[709800],eng,,,43219002,23615331,11,3.75,112,"[{'count': '153', 'name': 'to-read'}, {'count'...",Kindle Edition,279,Amy Raby,2014,Marius believes himself to be an ordinary smal...,"[{'author_id': '6462472', 'role': ''}]","[15738275, 23557009, 17225973, 15750539, 23996..."
10,"Nemesis (Southern Comfort, #4)",[521302],eng,,,25481836,18138604,10,4.09,162,"[{'count': '647', 'name': 'to-read'}, {'count'...",Kindle Edition,524,,2013,Chosen as one of the Best Reads of 2013 by Kin...,"[{'author_id': '6480066', 'role': ''}]","[21917872, 25698219, 17068443, 16005563, 16299..."
14,Love Means... Patience,[629995],eng,1627988912,9781627988919,41377166,22499086,1,4.11,6,"[{'count': '201', 'name': 'to-read'}, {'count'...",Paperback,200,Dreamspinner Press,2014,A Love Means... Story\nYears after his dischar...,"[{'author_id': '2953781', 'role': ''}]","[18386330, 22072041, 22041415, 21454390, 25278..."
15,"A.I. Revolution, Vol. 1",[910384],eng,1933617640,9781933617640,2256459,2250580,7,3.44,46,"[{'count': '15', 'name': 'to-read'}, {'count':...",Paperback Manga,206,Go! Comi,2007,"Like everyone else in the future, Sui's used t...","[{'author_id': '1015982', 'role': ''}]",[]


In [36]:
def clean_missing(x):
    if x is None:
        return np.nan
    if isinstance(x, str) and x.strip() == "":
        return np.nan
    if isinstance(x, list) and len(x) == 0:
        return np.nan
    return x


def unique_list(series):
    seen = set()
    result = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        key = str(x)

        if key not in seen:
            seen.add(key)
            result.append(x)

    return result


def first_non_missing(series):
    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        return x

    return np.nan


def longest_text(series):
    texts = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, str) and x.strip() != "":
            texts.append(x.strip())

    if len(texts) == 0:
        return np.nan

    return max(texts, key=len)


def max_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.max()


def mean_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.mean()

In [37]:
numeric_cols = [
    "text_reviews_count",
    "average_rating",
    "ratings_count",
    "num_pages",
    "publication_year"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [38]:
agg_dict = {
    # Main identity fields
    "clean_title": first_non_missing,
    "authors": first_non_missing,
    "series": unique_list,
    "language_code_final": first_non_missing,

    # IDs for API calls / joins
    "isbn": unique_list,
    "isbn13": unique_list,
    "book_id": unique_list,

    # Edition-level metadata
    "publisher": unique_list,
    "publication_year": unique_list,

    # Book-level text/features
    "description": longest_text,
    "popular_shelves": first_non_missing,
    "similar_books": unique_list,

    # Numeric summary features
    "text_reviews_count": max_numeric,
    "ratings_count": max_numeric,
    "average_rating": mean_numeric,
    "num_pages": max_numeric,
}

# keep only columns that exist in df
agg_dict = {col: func for col, func in agg_dict.items() if col in df.columns}

books_work_df = (
    df.groupby("work_id", dropna=False)
    .agg(agg_dict)
    .reset_index()
)

In [39]:
rename_dict = {
    "series": "series_list",
    "isbn": "isbn_list",
    "isbn13": "isbn13_list",
    "book_id": "book_id_list",
    "publisher": "publisher_list",
    "publication_year": "publication_year_list",
    "similar_books": "similar_books_list",
}

books_work_df = books_work_df.rename(
    columns={k: v for k, v in rename_dict.items() if k in books_work_df.columns}
)

In [40]:
books_work_df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,375,"The Bride Stripped Bare (Bride Trilogy, #1)","[{'author_id': '36165', 'role': ''}]",[[517518]],eng,"[0060591889, 0060735163, 0007170866, 006219147...","[9780060591885, 9780060735166, 9780007170869, ...","[64179, 888967, 888965, 15819527, 70936, 64185]","[Harper Perennial, Fourth Estate (GB), Fourth ...","[2012.0, 2004.0, 2003.0]","A woman disappears, leaving behind an incendia...","[{'count': '167', 'name': 'to-read'}, {'count'...","[[956897, 6251923, 1279734, 15725003, 1975139,...",399,3577,3.12,400.0
1,2363,Bookends,"[{'author_id': '12915', 'role': ''}]",[],eng,"[0767907809, 3453869796, 9780767907, 076790781...","[9780767907804, 9783453869790, 9780767907811, ...","[31107, 56209, 12863586, 22649, 22655]","[Broadway Books, Heyne, Penguin Books]","[2002.0, 2003.0, 2000.0]",On the heels of her national bestsellers Jemim...,"[{'count': '11184', 'name': 'to-read'}, {'coun...","[[33768, 196404, 9298, 161344, 3632, 241846, 2...",758,33365,3.70,426.0
2,2384,Fantasy Lover,"[{'author_id': '4430', 'role': ''}]","[[772503, 559092]]",eng,"[0749907436, 0312979975, 0739423150, 142993897...","[9780749907433, 9780312979973, 9780739423158, ...","[21051499, 84136, 11851761, 3331258, 470916, 5...","[Not Avail, St. Martin's Press, Piatkus Books,...","[2005.0, 2002.0, 2010.0, 2006.0, 2011.0]","See alternate cover edition: \nDear Reader,\n...","[{'count': '1405', 'name': 'romance'}, {'count...","[[84158, 61823, 2712967, 7715664, 84176, 70983...",2604,71242,4.17,368.0
3,2538,Ain't She Sweet,"[{'author_id': '41313', 'role': ''}]",[],eng,"[0749935081, 0061801216, 0061032085, 006058977...","[9780749935085, 9780061801211, 9780061032080, ...","[373606, 10167815, 790144, 3007864, 7149569]","[Piatkus, William Morrow, Avon, WmMorrow, Harp...","[2006.0, 2009.0, 2005.0, 2004.0]",Ain't She Sweet? Not exactly . . .\nThe girl e...,"[{'count': '5758', 'name': 'to-read'}, {'count...","[[60222, 129617, 1756703, 33756, 2884065, 7309...",738,16126,4.05,528.0
4,2539,Just Imagine,"[{'author_id': '41313', 'role': ''}]",[],eng,"[0060522534, 0066212324, 006179600X, 0380808307]","[9780060522537, 9780066212326, 9780061796005, ...","[7196046, 1396919, 10164151, 117366]","[HarperCollins Publishers, HarperCollins, Will...","[2002.0, 2009.0, 2001.0]",Rewrite of RISEN GLORY\nThe War Between the St...,"[{'count': '1913', 'name': 'to-read'}, {'count...","[[528414, 36570, 731539, 222757, 10762173, 124...",209,5578,3.72,384.0


In [41]:
df = books_work_df.copy()

In [42]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,375,"The Bride Stripped Bare (Bride Trilogy, #1)","[{'author_id': '36165', 'role': ''}]",[[517518]],eng,"[0060591889, 0060735163, 0007170866, 006219147...","[9780060591885, 9780060735166, 9780007170869, ...","[64179, 888967, 888965, 15819527, 70936, 64185]","[Harper Perennial, Fourth Estate (GB), Fourth ...","[2012.0, 2004.0, 2003.0]","A woman disappears, leaving behind an incendia...","[{'count': '167', 'name': 'to-read'}, {'count'...","[[956897, 6251923, 1279734, 15725003, 1975139,...",399,3577,3.12,400.0
1,2363,Bookends,"[{'author_id': '12915', 'role': ''}]",[],eng,"[0767907809, 3453869796, 9780767907, 076790781...","[9780767907804, 9783453869790, 9780767907811, ...","[31107, 56209, 12863586, 22649, 22655]","[Broadway Books, Heyne, Penguin Books]","[2002.0, 2003.0, 2000.0]",On the heels of her national bestsellers Jemim...,"[{'count': '11184', 'name': 'to-read'}, {'coun...","[[33768, 196404, 9298, 161344, 3632, 241846, 2...",758,33365,3.70,426.0
2,2384,Fantasy Lover,"[{'author_id': '4430', 'role': ''}]","[[772503, 559092]]",eng,"[0749907436, 0312979975, 0739423150, 142993897...","[9780749907433, 9780312979973, 9780739423158, ...","[21051499, 84136, 11851761, 3331258, 470916, 5...","[Not Avail, St. Martin's Press, Piatkus Books,...","[2005.0, 2002.0, 2010.0, 2006.0, 2011.0]","See alternate cover edition: \nDear Reader,\n...","[{'count': '1405', 'name': 'romance'}, {'count...","[[84158, 61823, 2712967, 7715664, 84176, 70983...",2604,71242,4.17,368.0
3,2538,Ain't She Sweet,"[{'author_id': '41313', 'role': ''}]",[],eng,"[0749935081, 0061801216, 0061032085, 006058977...","[9780749935085, 9780061801211, 9780061032080, ...","[373606, 10167815, 790144, 3007864, 7149569]","[Piatkus, William Morrow, Avon, WmMorrow, Harp...","[2006.0, 2009.0, 2005.0, 2004.0]",Ain't She Sweet? Not exactly . . .\nThe girl e...,"[{'count': '5758', 'name': 'to-read'}, {'count...","[[60222, 129617, 1756703, 33756, 2884065, 7309...",738,16126,4.05,528.0
4,2539,Just Imagine,"[{'author_id': '41313', 'role': ''}]",[],eng,"[0060522534, 0066212324, 006179600X, 0380808307]","[9780060522537, 9780066212326, 9780061796005, ...","[7196046, 1396919, 10164151, 117366]","[HarperCollins Publishers, HarperCollins, Will...","[2002.0, 2009.0, 2001.0]",Rewrite of RISEN GLORY\nThe War Between the St...,"[{'count': '1913', 'name': 'to-read'}, {'count...","[[528414, 36570, 731539, 222757, 10762173, 124...",209,5578,3.72,384.0


In [43]:
df.shape

(130699, 17)

In [44]:
df.isna().sum()

work_id                      0
clean_title                  0
authors                      0
series_list                  0
language_code_final          0
isbn_list                    0
isbn13_list                  0
book_id_list                 0
publisher_list               0
publication_year_list        0
description               4021
popular_shelves              0
similar_books_list           0
text_reviews_count           0
ratings_count                0
average_rating               0
num_pages                23399
dtype: int64

In [45]:
df.shape

(130699, 17)

In [46]:
df = df[df["description"].notna()].copy()

In [47]:
df.isna().sum()

work_id                      0
clean_title                  0
authors                      0
series_list                  0
language_code_final          0
isbn_list                    0
isbn13_list                  0
book_id_list                 0
publisher_list               0
publication_year_list        0
description                  0
popular_shelves              0
similar_books_list           0
text_reviews_count           0
ratings_count                0
average_rating               0
num_pages                22339
dtype: int64

In [48]:
df.shape

(126678, 17)

In [49]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,375,"The Bride Stripped Bare (Bride Trilogy, #1)","[{'author_id': '36165', 'role': ''}]",[[517518]],eng,"[0060591889, 0060735163, 0007170866, 006219147...","[9780060591885, 9780060735166, 9780007170869, ...","[64179, 888967, 888965, 15819527, 70936, 64185]","[Harper Perennial, Fourth Estate (GB), Fourth ...","[2012.0, 2004.0, 2003.0]","A woman disappears, leaving behind an incendia...","[{'count': '167', 'name': 'to-read'}, {'count'...","[[956897, 6251923, 1279734, 15725003, 1975139,...",399,3577,3.12,400.0
1,2363,Bookends,"[{'author_id': '12915', 'role': ''}]",[],eng,"[0767907809, 3453869796, 9780767907, 076790781...","[9780767907804, 9783453869790, 9780767907811, ...","[31107, 56209, 12863586, 22649, 22655]","[Broadway Books, Heyne, Penguin Books]","[2002.0, 2003.0, 2000.0]",On the heels of her national bestsellers Jemim...,"[{'count': '11184', 'name': 'to-read'}, {'coun...","[[33768, 196404, 9298, 161344, 3632, 241846, 2...",758,33365,3.70,426.0
2,2384,Fantasy Lover,"[{'author_id': '4430', 'role': ''}]","[[772503, 559092]]",eng,"[0749907436, 0312979975, 0739423150, 142993897...","[9780749907433, 9780312979973, 9780739423158, ...","[21051499, 84136, 11851761, 3331258, 470916, 5...","[Not Avail, St. Martin's Press, Piatkus Books,...","[2005.0, 2002.0, 2010.0, 2006.0, 2011.0]","See alternate cover edition: \nDear Reader,\n...","[{'count': '1405', 'name': 'romance'}, {'count...","[[84158, 61823, 2712967, 7715664, 84176, 70983...",2604,71242,4.17,368.0
3,2538,Ain't She Sweet,"[{'author_id': '41313', 'role': ''}]",[],eng,"[0749935081, 0061801216, 0061032085, 006058977...","[9780749935085, 9780061801211, 9780061032080, ...","[373606, 10167815, 790144, 3007864, 7149569]","[Piatkus, William Morrow, Avon, WmMorrow, Harp...","[2006.0, 2009.0, 2005.0, 2004.0]",Ain't She Sweet? Not exactly . . .\nThe girl e...,"[{'count': '5758', 'name': 'to-read'}, {'count...","[[60222, 129617, 1756703, 33756, 2884065, 7309...",738,16126,4.05,528.0
4,2539,Just Imagine,"[{'author_id': '41313', 'role': ''}]",[],eng,"[0060522534, 0066212324, 006179600X, 0380808307]","[9780060522537, 9780066212326, 9780061796005, ...","[7196046, 1396919, 10164151, 117366]","[HarperCollins Publishers, HarperCollins, Will...","[2002.0, 2009.0, 2001.0]",Rewrite of RISEN GLORY\nThe War Between the St...,"[{'count': '1913', 'name': 'to-read'}, {'count...","[[528414, 36570, 731539, 222757, 10762173, 124...",209,5578,3.72,384.0


In [50]:
df = df.drop(columns=["language_code_final", "publication_year_list"])

In [51]:
df.shape

(126678, 15)

In [52]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,375,"The Bride Stripped Bare (Bride Trilogy, #1)","[{'author_id': '36165', 'role': ''}]",[[517518]],"[0060591889, 0060735163, 0007170866, 006219147...","[9780060591885, 9780060735166, 9780007170869, ...","[64179, 888967, 888965, 15819527, 70936, 64185]","[Harper Perennial, Fourth Estate (GB), Fourth ...","A woman disappears, leaving behind an incendia...","[{'count': '167', 'name': 'to-read'}, {'count'...","[[956897, 6251923, 1279734, 15725003, 1975139,...",399,3577,3.12,400.0
1,2363,Bookends,"[{'author_id': '12915', 'role': ''}]",[],"[0767907809, 3453869796, 9780767907, 076790781...","[9780767907804, 9783453869790, 9780767907811, ...","[31107, 56209, 12863586, 22649, 22655]","[Broadway Books, Heyne, Penguin Books]",On the heels of her national bestsellers Jemim...,"[{'count': '11184', 'name': 'to-read'}, {'coun...","[[33768, 196404, 9298, 161344, 3632, 241846, 2...",758,33365,3.70,426.0
2,2384,Fantasy Lover,"[{'author_id': '4430', 'role': ''}]","[[772503, 559092]]","[0749907436, 0312979975, 0739423150, 142993897...","[9780749907433, 9780312979973, 9780739423158, ...","[21051499, 84136, 11851761, 3331258, 470916, 5...","[Not Avail, St. Martin's Press, Piatkus Books,...","See alternate cover edition: \nDear Reader,\n...","[{'count': '1405', 'name': 'romance'}, {'count...","[[84158, 61823, 2712967, 7715664, 84176, 70983...",2604,71242,4.17,368.0
3,2538,Ain't She Sweet,"[{'author_id': '41313', 'role': ''}]",[],"[0749935081, 0061801216, 0061032085, 006058977...","[9780749935085, 9780061801211, 9780061032080, ...","[373606, 10167815, 790144, 3007864, 7149569]","[Piatkus, William Morrow, Avon, WmMorrow, Harp...",Ain't She Sweet? Not exactly . . .\nThe girl e...,"[{'count': '5758', 'name': 'to-read'}, {'count...","[[60222, 129617, 1756703, 33756, 2884065, 7309...",738,16126,4.05,528.0
4,2539,Just Imagine,"[{'author_id': '41313', 'role': ''}]",[],"[0060522534, 0066212324, 006179600X, 0380808307]","[9780060522537, 9780066212326, 9780061796005, ...","[7196046, 1396919, 10164151, 117366]","[HarperCollins Publishers, HarperCollins, Will...",Rewrite of RISEN GLORY\nThe War Between the St...,"[{'count': '1913', 'name': 'to-read'}, {'count...","[[528414, 36570, 731539, 222757, 10762173, 124...",209,5578,3.72,384.0


In [53]:
df["clean_title"] = (
    df["clean_title"]
    .astype(str)
    .str.replace(r"\s*\([^)]*#\d+[^)]*\)", "", regex=True)
    .str.strip()
)

In [54]:
def extract_author_ids(authors):
    if not isinstance(authors, list):
        return []

    return [
        str(a.get("author_id"))
        for a in authors
        if isinstance(a, dict) and a.get("author_id")
    ]

df["author_ids"] = df["authors"].apply(extract_author_ids)

In [55]:
def flatten_nested_list(x):
    if not isinstance(x, list):
        return []

    flat = []

    for item in x:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)

    # remove missing/empty values and dedupe
    result = []
    seen = set()

    for item in flat:
        if item is None:
            continue

        item_str = str(item).strip()

        if item_str == "" or item_str.lower() in {"nan", "none"}:
            continue

        if item_str not in seen:
            seen.add(item_str)
            result.append(item)

    return result

In [56]:
for col in ["series_list", "similar_books_list"]:
    if col in df.columns:
        df[col] = df[col].apply(flatten_nested_list)

In [57]:
numeric_cols = [
    "text_reviews_count",
    "ratings_count",
    "average_rating",
    "num_pages"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [58]:
df["num_pages_missing"] = df["num_pages"].isna().astype(int)

In [59]:
for i, shelves in enumerate(df["popular_shelves"].head(10), start=1):
    print(f"\nRow {i}:")
    print(shelves)


Row 1:
[{'count': '167', 'name': 'to-read'}, {'count': '22', 'name': 'contemporary'}, {'count': '18', 'name': 'australian'}, {'count': '14', 'name': 'favorites'}, {'count': '11', 'name': 'default'}, {'count': '10', 'name': 'currently-reading'}, {'count': '10', 'name': 'fiction'}, {'count': '9', 'name': 'book-club'}, {'count': '8', 'name': 'australia'}, {'count': '8', 'name': 'favourites'}, {'count': '7', 'name': 'abandoned'}, {'count': '7', 'name': 'literature'}, {'count': '6', 'name': 'erotica'}, {'count': '6', 'name': 'kindle'}, {'count': '6', 'name': 'novels'}, {'count': '6', 'name': 'drama'}, {'count': '5', 'name': 'my-library'}, {'count': '5', 'name': 'series'}, {'count': '5', 'name': 'bought'}, {'count': '5', 'name': 'contemporary-fiction'}, {'count': '5', 'name': 'steamy'}, {'count': '5', 'name': 'i-own'}, {'count': '5', 'name': 'sexuality'}, {'count': '5', 'name': 'own-it'}, {'count': '4', 'name': 'read-in-2016'}, {'count': '4', 'name': 'owned-books'}, {'count': '4', 'name': '

In [60]:
int_df = df[['popular_shelves']]

In [61]:
int_df.head()

,popular_shelves
0,"[{'count': '167', 'name': 'to-read'}, {'count'..."
1,"[{'count': '11184', 'name': 'to-read'}, {'coun..."
2,"[{'count': '1405', 'name': 'romance'}, {'count..."
3,"[{'count': '5758', 'name': 'to-read'}, {'count..."
4,"[{'count': '1913', 'name': 'to-read'}, {'count..."


In [62]:
int_df.shape

(126678, 1)

In [63]:
int_df.to_csv('../data/intermediate/romance_shelves.csv', index=False)

In [64]:
# TAG NORMALIZATION MAP ---> BUILT WITH CLAUDE
# instead of calling Claude or gpt's api or even using a model from HuggingFace, this seemed to be the fastest and the cheapest way
shelves = pd.read_csv('../data/intermediate/romance_tag_mapping.csv')

In [65]:
shelves.head(20)

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,70374047,125809
1,currently-reading,status,drop,3859891,101181
2,romance,romance,keep,2853701,117061
3,favorites,review,drop,1277914,58649
4,contemporary,contemporary,keep,861416,69431
5,kindle,ownership,drop,757738,75591
6,series,noise,drop,693029,60490
7,m-m,lgbt,keep,657564,19726
8,contemporary-romance,contemporary-romance,keep,592143,57608
9,historical-romance,historical-romance,keep,565130,17739


In [66]:
vocab_df = shelves.copy()

In [67]:
tag_map = dict(
    zip(
        vocab_df.loc[vocab_df["action"] == "keep", "raw_tag"],
        vocab_df.loc[vocab_df["action"] == "keep", "clean_tag"]
    )
)

In [68]:
vocab_df.head()

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,70374047,125809
1,currently-reading,status,drop,3859891,101181
2,romance,romance,keep,2853701,117061
3,favorites,review,drop,1277914,58649
4,contemporary,contemporary,keep,861416,69431


In [69]:
def replace_shelf_tags(shelves):
    # if shelves got saved as a string, convert it back to list
    if isinstance(shelves, str):
        try:
            shelves = ast.literal_eval(shelves)
        except Exception:
            return []

    if not isinstance(shelves, list):
        return []

    cleaned_shelves = []

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        raw_tag = shelf.get("name")

        # only replace if action == keep and raw_tag exists in tag_map
        if raw_tag in tag_map:
            new_shelf = shelf.copy()
            new_shelf["name"] = tag_map[raw_tag]
            cleaned_shelves.append(new_shelf)

    return cleaned_shelves

In [70]:
df["popular_shelves_cleaned"] = df["popular_shelves"].apply(replace_shelf_tags)

In [71]:
df[["popular_shelves", "popular_shelves_cleaned"]].head()

,popular_shelves,popular_shelves_cleaned
0,"[{'count': '167', 'name': 'to-read'}, {'count'...","[{'count': '22', 'name': 'contemporary'}, {'co..."
1,"[{'count': '11184', 'name': 'to-read'}, {'coun...","[{'count': '709', 'name': 'romance'}, {'count'..."
2,"[{'count': '1405', 'name': 'romance'}, {'count...","[{'count': '1405', 'name': 'romance'}, {'count..."
3,"[{'count': '5758', 'name': 'to-read'}, {'count...","[{'count': '749', 'name': 'romance'}, {'count'..."
4,"[{'count': '1913', 'name': 'to-read'}, {'count...","[{'count': '282', 'name': 'romance'}, {'count'..."


In [72]:
print("Original popular_shelves:")
print(df.loc[df.index[0], "popular_shelves"])

print("\nCleaned popular_shelves:")
print(df.loc[df.index[0], "popular_shelves_cleaned"])

Original popular_shelves:
[{'count': '167', 'name': 'to-read'}, {'count': '22', 'name': 'contemporary'}, {'count': '18', 'name': 'australian'}, {'count': '14', 'name': 'favorites'}, {'count': '11', 'name': 'default'}, {'count': '10', 'name': 'currently-reading'}, {'count': '10', 'name': 'fiction'}, {'count': '9', 'name': 'book-club'}, {'count': '8', 'name': 'australia'}, {'count': '8', 'name': 'favourites'}, {'count': '7', 'name': 'abandoned'}, {'count': '7', 'name': 'literature'}, {'count': '6', 'name': 'erotica'}, {'count': '6', 'name': 'kindle'}, {'count': '6', 'name': 'novels'}, {'count': '6', 'name': 'drama'}, {'count': '5', 'name': 'my-library'}, {'count': '5', 'name': 'series'}, {'count': '5', 'name': 'bought'}, {'count': '5', 'name': 'contemporary-fiction'}, {'count': '5', 'name': 'steamy'}, {'count': '5', 'name': 'i-own'}, {'count': '5', 'name': 'sexuality'}, {'count': '5', 'name': 'own-it'}, {'count': '4', 'name': 'read-in-2016'}, {'count': '4', 'name': 'owned-books'}, {'coun

In [73]:
def merge_duplicate_shelf_tags(shelves):
    if not isinstance(shelves, list):
        return []

    merged = OrderedDict()

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        name = shelf.get("name")
        count = shelf.get("count", 0)

        if name is None or str(name).strip() == "":
            continue

        name = str(name).strip()

        try:
            count = int(count)
        except Exception:
            count = 0

        if name not in merged:
            merged[name] = count
        else:
            merged[name] += count

    return [
        {"count": str(count), "name": name}
        for name, count in merged.items()
    ]

In [74]:
df["popular_shelves_cleaned"] = df["popular_shelves_cleaned"].apply(
    merge_duplicate_shelf_tags
)

In [75]:
print(df.loc[df.index[1], "popular_shelves_cleaned"])

[{'count': '1050', 'name': 'romance'}, {'count': '254', 'name': 'fiction'}, {'count': '41', 'name': 'contemporary'}, {'count': '53', 'name': 'adult'}, {'count': '34', 'name': 'british-history'}, {'count': '13', 'name': 'contemporary-romance'}, {'count': '11', 'name': 'friendship'}, {'count': '14', 'name': 'humor'}, {'count': '13', 'name': 'womens-history'}, {'count': '4', 'name': 'realistic-fiction'}, {'count': '3', 'name': 'lgbt'}]


In [76]:
for shelf in df.loc[df.index[10], "popular_shelves_cleaned"]:
    print(shelf)

{'count': '200', 'name': 'romance'}
{'count': '49', 'name': 'contemporary'}
{'count': '81', 'name': 'war'}
{'count': '33', 'name': 'contemporary-romance'}
{'count': '28', 'name': 'harlequin'}
{'count': '18', 'name': 'fiction'}
{'count': '22', 'name': 'thriller'}
{'count': '7', 'name': 'holiday'}
{'count': '11', 'name': 'adventure'}
{'count': '10', 'name': 'mystery'}
{'count': '6', 'name': 'adult'}


In [77]:
df.shape

(126678, 18)

In [78]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,author_ids,num_pages_missing,popular_shelves_cleaned
0,375,The Bride Stripped Bare,"[{'author_id': '36165', 'role': ''}]",[517518],"[0060591889, 0060735163, 0007170866, 006219147...","[9780060591885, 9780060735166, 9780007170869, ...","[64179, 888967, 888965, 15819527, 70936, 64185]","[Harper Perennial, Fourth Estate (GB), Fourth ...","A woman disappears, leaving behind an incendia...","[{'count': '167', 'name': 'to-read'}, {'count'...","[956897, 6251923, 1279734, 15725003, 1975139, ...",399,3577,3.12,400.0,[36165],0,"[{'count': '22', 'name': 'contemporary'}, {'co..."
1,2363,Bookends,"[{'author_id': '12915', 'role': ''}]",[],"[0767907809, 3453869796, 9780767907, 076790781...","[9780767907804, 9783453869790, 9780767907811, ...","[31107, 56209, 12863586, 22649, 22655]","[Broadway Books, Heyne, Penguin Books]",On the heels of her national bestsellers Jemim...,"[{'count': '11184', 'name': 'to-read'}, {'coun...","[33768, 196404, 9298, 161344, 3632, 241846, 20...",758,33365,3.70,426.0,[12915],0,"[{'count': '1050', 'name': 'romance'}, {'count..."
2,2384,Fantasy Lover,"[{'author_id': '4430', 'role': ''}]","[772503, 559092]","[0749907436, 0312979975, 0739423150, 142993897...","[9780749907433, 9780312979973, 9780739423158, ...","[21051499, 84136, 11851761, 3331258, 470916, 5...","[Not Avail, St. Martin's Press, Piatkus Books,...","See alternate cover edition: \nDear Reader,\n...","[{'count': '1405', 'name': 'romance'}, {'count...","[84158, 61823, 2712967, 7715664, 84176, 709830...",2604,71242,4.17,368.0,[4430],0,"[{'count': '1573', 'name': 'romance'}, {'count..."
3,2538,Ain't She Sweet,"[{'author_id': '41313', 'role': ''}]",[],"[0749935081, 0061801216, 0061032085, 006058977...","[9780749935085, 9780061801211, 9780061032080, ...","[373606, 10167815, 790144, 3007864, 7149569]","[Piatkus, William Morrow, Avon, WmMorrow, Harp...",Ain't She Sweet? Not exactly . . .\nThe girl e...,"[{'count': '5758', 'name': 'to-read'}, {'count...","[60222, 129617, 1756703, 33756, 2884065, 73093...",738,16126,4.05,528.0,[41313],0,"[{'count': '1124', 'name': 'romance'}, {'count..."
4,2539,Just Imagine,"[{'author_id': '41313', 'role': ''}]",[],"[0060522534, 0066212324, 006179600X, 0380808307]","[9780060522537, 9780066212326, 9780061796005, ...","[7196046, 1396919, 10164151, 117366]","[HarperCollins Publishers, HarperCollins, Will...",Rewrite of RISEN GLORY\nThe War Between the St...,"[{'count': '1913', 'name': 'to-read'}, {'count...","[528414, 36570, 731539, 222757, 10762173, 1240...",209,5578,3.72,384.0,[41313],0,"[{'count': '377', 'name': 'romance'}, {'count'..."


In [79]:
df.columns

Index(['work_id', 'clean_title', 'authors', 'series_list', 'isbn_list',
       'isbn13_list', 'book_id_list', 'publisher_list', 'description',
       'popular_shelves', 'similar_books_list', 'text_reviews_count',
       'ratings_count', 'average_rating', 'num_pages', 'author_ids',
       'num_pages_missing', 'popular_shelves_cleaned'],
      dtype='str')

In [80]:
df = df[['work_id', 'clean_title', 'author_ids', 'series_list', 'isbn_list', 
         'isbn13_list', 'book_id_list', 'publisher_list', 'description', 
         'similar_books_list', 'text_reviews_count', 'ratings_count', 
         'average_rating', 'num_pages', 'num_pages_missing', 'popular_shelves_cleaned']]

In [81]:
df.head()

,work_id,clean_title,author_ids,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,num_pages_missing,popular_shelves_cleaned
0,375,The Bride Stripped Bare,[36165],[517518],"[0060591889, 0060735163, 0007170866, 006219147...","[9780060591885, 9780060735166, 9780007170869, ...","[64179, 888967, 888965, 15819527, 70936, 64185]","[Harper Perennial, Fourth Estate (GB), Fourth ...","A woman disappears, leaving behind an incendia...","[956897, 6251923, 1279734, 15725003, 1975139, ...",399,3577,3.12,400.0,0,"[{'count': '22', 'name': 'contemporary'}, {'co..."
1,2363,Bookends,[12915],[],"[0767907809, 3453869796, 9780767907, 076790781...","[9780767907804, 9783453869790, 9780767907811, ...","[31107, 56209, 12863586, 22649, 22655]","[Broadway Books, Heyne, Penguin Books]",On the heels of her national bestsellers Jemim...,"[33768, 196404, 9298, 161344, 3632, 241846, 20...",758,33365,3.70,426.0,0,"[{'count': '1050', 'name': 'romance'}, {'count..."
2,2384,Fantasy Lover,[4430],"[772503, 559092]","[0749907436, 0312979975, 0739423150, 142993897...","[9780749907433, 9780312979973, 9780739423158, ...","[21051499, 84136, 11851761, 3331258, 470916, 5...","[Not Avail, St. Martin's Press, Piatkus Books,...","See alternate cover edition: \nDear Reader,\n...","[84158, 61823, 2712967, 7715664, 84176, 709830...",2604,71242,4.17,368.0,0,"[{'count': '1573', 'name': 'romance'}, {'count..."
3,2538,Ain't She Sweet,[41313],[],"[0749935081, 0061801216, 0061032085, 006058977...","[9780749935085, 9780061801211, 9780061032080, ...","[373606, 10167815, 790144, 3007864, 7149569]","[Piatkus, William Morrow, Avon, WmMorrow, Harp...",Ain't She Sweet? Not exactly . . .\nThe girl e...,"[60222, 129617, 1756703, 33756, 2884065, 73093...",738,16126,4.05,528.0,0,"[{'count': '1124', 'name': 'romance'}, {'count..."
4,2539,Just Imagine,[41313],[],"[0060522534, 0066212324, 006179600X, 0380808307]","[9780060522537, 9780066212326, 9780061796005, ...","[7196046, 1396919, 10164151, 117366]","[HarperCollins Publishers, HarperCollins, Will...",Rewrite of RISEN GLORY\nThe War Between the St...,"[528414, 36570, 731539, 222757, 10762173, 1240...",209,5578,3.72,384.0,0,"[{'count': '377', 'name': 'romance'}, {'count'..."


In [82]:
df.shape

(126678, 16)

In [83]:
df.to_csv('../data/intermediate/books-cleaned/romance.csv', index=False)